In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# ==================================================
# CONNECT DB
# ==================================================

con = duckdb.connect(
    "../data/warehouse/ecommerce.duckdb",
    read_only=True
)

In [6]:
# ==================================================
# INSIGHT 1
# FUNNEL ANALYSIS
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 1: CUSTOMER FUNNEL LEAKAGE")
print("=" * 80)

df_funnel = con.execute("""
SELECT *
FROM mart_funnel
ORDER BY stage_name
""").df()

print(df_funnel)

view_users = df_funnel.loc[
    df_funnel["stage_name"] == "View",
    "user_count"
].iloc[0]

cart_users = df_funnel.loc[
    df_funnel["stage_name"] == "Cart",
    "user_count"
].iloc[0]

purchase_users = df_funnel.loc[
    df_funnel["stage_name"] == "Purchase",
    "user_count"
].iloc[0]

view_to_cart = cart_users / view_users
cart_to_purchase = purchase_users / cart_users
overall_cr = purchase_users / view_users

print(f"""
[FINDING]

Users Viewed     : {view_users:,.0f}
Users Carted     : {cart_users:,.0f}
Users Purchased  : {purchase_users:,.0f}

View → Cart CR       : {view_to_cart:.2%}
Cart → Purchase CR   : {cart_to_purchase:.2%}
Overall CR           : {overall_cr:.2%}

[BUSINESS IMPACT]

Traffic đang rất lớn nhưng một lượng lớn user
rời khỏi hành trình trước khi hoàn tất mua hàng.

[RECOMMENDATION]

1. Tối ưu PDP (Product Detail Page)
2. Tối ưu Checkout Flow
3. Remarketing cho nhóm Cart Abandonment
""")



INSIGHT 1: CUSTOMER FUNNEL LEAKAGE
  stage_name  user_count  conversion_rate
0       Cart      826317           0.2236
1   Purchase      441638           0.1195
2       View     3695598           1.0000

[FINDING]

Users Viewed     : 3,695,598
Users Carted     : 826,317
Users Purchased  : 441,638

View → Cart CR       : 22.36%
Cart → Purchase CR   : 53.45%
Overall CR           : 11.95%

[BUSINESS IMPACT]

Traffic đang rất lớn nhưng một lượng lớn user
rời khỏi hành trình trước khi hoàn tất mua hàng.

[RECOMMENDATION]

1. Tối ưu PDP (Product Detail Page)
2. Tối ưu Checkout Flow
3. Remarketing cho nhóm Cart Abandonment



In [7]:
# ==================================================
# INSIGHT 2
# RETENTION
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 2: CUSTOMER RETENTION")
print("=" * 80)

df_retention = con.execute("""
SELECT
    cohort_index,
    AVG(retention_rate) as retention_rate
FROM mart_retention
GROUP BY 1
ORDER BY 1
""").df()

print(df_retention)

week0 = df_retention.iloc[0]["retention_rate"]

week_last = df_retention.iloc[
    len(df_retention)-1
]["retention_rate"]

print(f"""
[FINDING]

Week 0 Retention : {week0:.2%}
Last Week Retention : {week_last:.2%}

[BUSINESS IMPACT]

Khách hàng mới không quay lại đủ nhiều.

Doanh thu hiện tại đang phụ thuộc mạnh
vào việc liên tục mua traffic mới.

[RECOMMENDATION]

1. Loyalty Program
2. Retention Campaign
3. Personalized Offers
4. Email Remarketing
""")



INSIGHT 2: CUSTOMER RETENTION
   cohort_index  retention_rate
0             0         1.00000
1             1         0.30975
2             2         0.29150
3             3         0.26165
4             4         0.27320

[FINDING]

Week 0 Retention : 100.00%
Last Week Retention : 27.32%

[BUSINESS IMPACT]

Khách hàng mới không quay lại đủ nhiều.

Doanh thu hiện tại đang phụ thuộc mạnh
vào việc liên tục mua traffic mới.

[RECOMMENDATION]

1. Loyalty Program
2. Retention Campaign
3. Personalized Offers
4. Email Remarketing



In [2]:
# ==================================================
# INSIGHT 3
# CUSTOMER SEGMENTS
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 3: RFM CUSTOMER SEGMENTS")
print("=" * 80)

df_segment = con.execute("""
SELECT
    customer_segment,
    COUNT(*) as users,
    SUM(monetary) as revenue
FROM mart_customer_segments
GROUP BY 1
ORDER BY revenue DESC
""").df()

print(df_segment)

top_segment = df_segment.iloc[0]

print(f"""
[FINDING]

Top Revenue Segment:

{top_segment['customer_segment']}

Revenue:

{top_segment['revenue']:,.2f}

[BUSINESS IMPACT]

Một nhóm khách hàng nhỏ tạo ra
phần lớn doanh thu.

[RECOMMENDATION]

1. VIP Program
2. Exclusive Promotion
3. Personalized Experience
4. Churn Prevention Strategy
""")



INSIGHT 3: RFM CUSTOMER SEGMENTS
  customer_segment  users       revenue
0        Champions  57952  1.155088e+08
1          At Risk  60879  5.267731e+07
2  Loyal Customers  56123  4.174979e+07
3   Lost Customers  71135  2.306589e+07
4   About To Sleep  60666  2.109298e+07
5    New Customers  44855  1.325299e+07
6  Needs Attention  90028  7.845357e+06

[FINDING]

Top Revenue Segment:

Champions

Revenue:

115,508,762.62

[BUSINESS IMPACT]

Một nhóm khách hàng nhỏ tạo ra
phần lớn doanh thu.

[RECOMMENDATION]

1. VIP Program
2. Exclusive Promotion
3. Personalized Experience
4. Churn Prevention Strategy



In [9]:
# ==================================================
# INSIGHT 4
# CATEGORY PERFORMANCE
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 4: CATEGORY PERFORMANCE")
print("=" * 80)

df_category = con.execute("""
SELECT
    category_group,
    SUM(total_sessions) as sessions,
    SUM(cart_sessions) as carts,
    SUM(purchase_sessions) as purchases
FROM int_daily_category_metrics
GROUP BY 1
ORDER BY purchases DESC
""").df()

print(df_category)

print("""
[FINDING]

Một số category thu hút lượng lớn traffic
nhưng chuyển đổi thấp.

[BUSINESS IMPACT]

Traffic không đồng nghĩa với doanh thu.

Cần tập trung tối ưu nhóm category
có volume cao nhưng CR thấp.

[RECOMMENDATION]

1. Category-specific Promotion
2. Improve Product Assortment
3. Optimize Product Ranking
""")



INSIGHT 4: CATEGORY PERFORMANCE
   category_group   sessions     carts  purchases
0     electronics  5990736.0  887359.0   417810.0
1                  5108374.0  503945.0   208628.0
2      appliances  1643443.0  206156.0    88390.0
3       computers   933115.0   75073.0    30140.0
4         apparel   697283.0   33984.0    12659.0
5       furniture   523729.0   26990.0    10333.0
6            auto   307529.0   24618.0    10054.0
7    construction   251337.0   21471.0     7998.0
8            kids   262848.0   13359.0     5598.0
9     accessories   103340.0    5120.0     2019.0
10          sport    72296.0    3791.0     1368.0
11       medicine     6229.0     798.0      330.0
12     stationery     5192.0     423.0      159.0
13   country_yard     5884.0     220.0       60.0

[FINDING]

Một số category thu hút lượng lớn traffic
nhưng chuyển đổi thấp.

[BUSINESS IMPACT]

Traffic không đồng nghĩa với doanh thu.

Cần tập trung tối ưu nhóm category
có volume cao nhưng CR thấp.

[RECOMMENDATI

In [3]:
# ==================================================
# INSIGHT 5
# CAUSAL ANALYSIS
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 5: DIFFERENCE-IN-DIFFERENCES")
print("=" * 80)

df_did = con.execute("""
SELECT
    is_treatment,
    is_post,
    AVG(cart_to_purchase_cr) as avg_cr
FROM mart_causal_did
GROUP BY 1,2
ORDER BY 1,2
""").df()

print(df_did)

control_pre = df_did.iloc[0]["avg_cr"]
control_post = df_did.iloc[1]["avg_cr"]

treatment_pre = df_did.iloc[2]["avg_cr"]
treatment_post = df_did.iloc[3]["avg_cr"]

did_effect = (
    (treatment_post - treatment_pre)
    -
    (control_post - control_pre)
)

print(f"""
[FINDING]

Treatment Pre  : {treatment_pre:.4f}
Treatment Post : {treatment_post:.4f}

Control Pre    : {control_pre:.4f}
Control Post   : {control_post:.4f}

DID Effect     : {did_effect:.4f}

[BUSINESS IMPACT]

Ước lượng tác động thuần của thay đổi
chính sách lên tỷ lệ chuyển đổi.

[RECOMMENDATION]

Triển khai A/B Testing hoặc
Causal Experiment thực tế để xác nhận.
""")

# ==================================================
# SUMMARY
# ==================================================

print("\n")
print("=" * 80)
print("EXECUTIVE SUMMARY")
print("=" * 80)

print("""
1. Funnel đang thất thoát đáng kể trước bước Purchase

2. Retention còn thấp, tăng trưởng phụ thuộc traffic mới

3. Nhóm VIP tạo phần lớn doanh thu

4. Có sự khác biệt hiệu suất giữa các ngành hàng

5. Causal Analysis cho thấy thay đổi chính sách
   có thể tác động đến conversion rate

=> Ưu tiên chiến lược:
   Retention + Funnel Optimization + VIP Loyalty
""")



INSIGHT 5: DIFFERENCE-IN-DIFFERENCES
   is_treatment  is_post    avg_cr
0             0        0  1.900191
1             0        1  0.436370
2             1        0  0.784127
3             1        1  0.514127

[FINDING]

Treatment Pre  : 0.7841
Treatment Post : 0.5141

Control Pre    : 1.9002
Control Post   : 0.4364

DID Effect     : 1.1938

[BUSINESS IMPACT]

Ước lượng tác động thuần của thay đổi
chính sách lên tỷ lệ chuyển đổi.

[RECOMMENDATION]

Triển khai A/B Testing hoặc
Causal Experiment thực tế để xác nhận.



EXECUTIVE SUMMARY

1. Funnel đang thất thoát đáng kể trước bước Purchase

2. Retention còn thấp, tăng trưởng phụ thuộc traffic mới

3. Nhóm VIP tạo phần lớn doanh thu

4. Có sự khác biệt hiệu suất giữa các ngành hàng

5. Causal Analysis cho thấy thay đổi chính sách
   có thể tác động đến conversion rate

=> Ưu tiên chiến lược:
   Retention + Funnel Optimization + VIP Loyalty



In [4]:
con.close()